In [1]:
import pandas as pd
import numpy as np
# =========================================================
# READ SOURCE DATA
# =========================================================
pvt_df = pd.read_excel("Pvt draft.xlsx")
ptc_df = pd.read_excel("PTC draft.xlsx")

# =========================================================
# NORMALIZE PN
# =========================================================
pvt_df["PN AL78"] = pvt_df["PN AL78"].astype(str).str.strip()
ptc_df["PN AL78"] = ptc_df["PN AL78"].astype(str).str.strip()

# =========================================================
# AGGREGATION: OH
# =========================================================
oh_df = (
    pvt_df
    .groupby("PN AL78", as_index=False)
    .agg({"OH": "mean"})
    .rename(columns={"OH": "Avrg.OH CM"})
)

# =========================================================
# MAIN AGGREGATION
# =========================================================
main_df = (
    pvt_df
    .groupby("PN AL78", as_index=False)
    .agg({
        "PN used": "first",
        "PN.Final": "first",
        "Description": "first",
        "In Rcv Total": "mean",
        "Qty Resvered": "sum",
        "In Tr Total": "mean",
        "7 Months Prior": "sum",
        "OO CM": "sum",
        "OO NM": "sum",
        "OO N2M": "sum",
        "Curr. Month": "max",
        "Req CM": "max",
        "Req NM": "max",
    })
)

# =========================================================
# MERGE OH
# =========================================================
result_df = main_df.merge(oh_df, on="PN AL78", how="left")

# =========================================================
# RENAME
# =========================================================
result_df = result_df.rename(columns={
    "PN.Final": "PN No suffix",
    "PN used": "Long PN",
    "In Rcv Total": "Avrg. InRcv",
    "Qty Resvered": "Sum Qty Resvrd",
    "In Tr Total": "Avrg. InTr",
    "7 Months Prior": "Sum 7 Months Prior",
    "OO CM": "Sum OO CM",
    "OO NM": "Sum OO NM",
    "OO N2M": "Sum OO N2M",
    "Curr. Month": "Max.Curr. Month",
    "Req CM": "Max.Req. CM",
    "Req NM": "Max.Req. NM",
})

# =========================================================
# ADD EMPTY COLUMNS
# =========================================================
empty_cols = [f"Blank_{i}" for i in range(1, 7)]
insert_pos = result_df.columns.get_loc("Max.Req. NM") + 1
for i, col in enumerate(empty_cols):
    result_df.insert(insert_pos + i, col, np.nan)

# =========================================================
# FIX ESD TYPES
# =========================================================
ptc_df["ESD Mon"] = pd.to_numeric(ptc_df["ESD Mon"], errors="coerce")
ptc_df["ESD Year"] = pd.to_numeric(ptc_df["ESD Year"], errors="coerce")

# =========================================================
# FLAG ROWS WITH NO ESD
# =========================================================
ptc_df["No_ESD"] = ptc_df["ESD Mon"].isna() | ptc_df["ESD Year"].isna()

# =========================================================
# CREATE ESD PERIOD
# =========================================================
ptc_df["ESD_Period"] = (
    ptc_df["ESD Mon"].astype("Int64").astype(str)
    + "-"
    + ptc_df["ESD Year"].astype("Int64").astype(str)
)

# =========================================================
# DYNAMIC PERIOD LIST
# =========================================================
esd_periods = (
    ptc_df.loc[~ptc_df["No_ESD"], ["ESD Year", "ESD Mon"]]
    .drop_duplicates()
    .sort_values(["ESD Year", "ESD Mon"])
    .assign(
        ESD_Period=lambda x:
        x["ESD Mon"].astype(int).astype(str)
        + "-"
        + x["ESD Year"].astype(int).astype(str)
    )["ESD_Period"]
    .tolist()
)

# =========================================================
# AGGREGATE WITH ESD
# =========================================================
ptc_esd = (
    ptc_df[~ptc_df["No_ESD"]]
    .groupby(["PN AL78", "ESD_Period"], as_index=False)
    .agg(Qty_Resvrd=("Qty Resvered", "sum"))
)

# =========================================================
# AGGREGATE WITHOUT ESD  ⭐⭐ KEY PART ⭐⭐
# =========================================================
ptc_missing = (
    ptc_df[ptc_df["No_ESD"]]
    .groupby("PN AL78", as_index=False)
    .agg(Missing=("Qty Resvered", "sum"))
)

# =========================================================
# PIVOT
# =========================================================
ptc_pivot = (
    ptc_esd
    .pivot(index="PN AL78", columns="ESD_Period", values="Qty_Resvrd")
    .reset_index()
)

# Ensure all months exist
for p in esd_periods:
    if p not in ptc_pivot.columns:
        ptc_pivot[p] = 0

ptc_pivot = ptc_pivot[["PN AL78"] + esd_periods]

# =========================================================
# MERGE MISSING INTO PIVOT
# =========================================================
ptc_pivot = ptc_pivot.merge(ptc_missing, on="PN AL78", how="left")
ptc_pivot["Missing"] = ptc_pivot["Missing"].fillna(0)

# =========================================================
# GRAND TOTAL CODE
# =========================================================
ptc_pivot["Grand Total"] = ptc_pivot[esd_periods].sum(axis=1) + ptc_pivot["Missing"]

# =========================================================
# MERGE BACK INTO RESULT
# =========================================================
result_df = result_df.merge(ptc_pivot, on="PN AL78", how="left")


In [2]:
# =========================================================
# ===== ADD BLANK COLUMNS + chk. Avail. (PANDAS) ==========
# =========================================================

insert_pos = result_df.columns.get_loc("Grand Total") + 1

# ---- Add 2 blank columns ----
result_df.insert(insert_pos, " ", np.nan)
result_df.insert(insert_pos + 1, "  ", np.nan)

# ---- Calculate chk. Avail. directly ----
supply = (
    result_df["Avrg.OH CM"].fillna(0)
    + result_df["Avrg. InRcv"].fillna(0)
    + result_df["Avrg. InTr"].fillna(0)
    + result_df["Sum 7 Months Prior"].fillna(0)
    + result_df["Sum OO CM"].fillna(0)
)


result_df.insert(
    insert_pos + 2,
    "chk. Avail.",
    np.where(
        result_df["Avrg.OH CM"] < result_df["Max.Curr. Month"],
        np.where(
            result_df["Max.Curr. Month"] <= supply,
            "OK",
            "NG"
        ),
        "OK"
    )
)


In [3]:
# =========================================================
# ===== ADD Balnc. COLUMN (PANDAS) ========================
# =========================================================

supply_cols = [
    "Avrg.OH CM",
    "Avrg. InRcv",
    "Avrg. InTr",
    "Sum 7 Months Prior",
    "Sum OO CM",
]

supply = result_df[supply_cols].fillna(0).sum(axis=1)

result_df["Balnc."] = supply - result_df["Max.Curr. Month"].fillna(0)


In [4]:
# =========================================================
# ===== ADD chk Resvrd COLUMN (PANDAS) ====================
# =========================================================

result_df.insert(
    result_df.columns.get_loc("Balnc.") + 1,
    "chk Resvrd",
    np.where(
        result_df["Grand Total"].fillna(0)
        == result_df["Sum Qty Resvrd"].fillna(0),
        "OK",
        "NG"
    )
)


In [5]:
# =========================================================
# DETECT CURRENT ESD PERIOD (EARLIEST)
# =========================================================
current_esd_col = (
    sorted(
        esd_periods,
        key=lambda x: (int(x.split("-")[1]), int(x.split("-")[0]))
    )[0]
)

# =========================================================
# chk avail.
# =========================================================
supply = (
    pd.to_numeric(result_df["Avrg.OH CM"], errors="coerce").fillna(0)
  + pd.to_numeric(result_df["Avrg. InRcv"], errors="coerce").fillna(0)
  + pd.to_numeric(result_df["Avrg. InTr"], errors="coerce").fillna(0)
  + pd.to_numeric(result_df.get(current_esd_col, 0), errors="coerce").fillna(0)
)

result_df["chk avail."] = np.where(
    supply >= pd.to_numeric(result_df["Max.Curr. Month"], errors="coerce").fillna(0),
    "OK",
    "check"
)



In [6]:
# =========================================================
# add empty column 'add'
# =========================================================

insert_pos = result_df.columns.get_loc("chk avail.") + 1
result_df.insert(insert_pos, "add", np.nan)


In [7]:
# =========================================================
# LOOKUP DN Prc FROM PTC NORMAL DRAFT
# =========================================================

ptc_normal_df = pd.read_excel("PTC Normal Draft.xlsx")

# Ensure PN AL78 is string for safe merge (letters allowed)
ptc_normal_df["PN AL78"] = ptc_normal_df["PN AL78"].astype(str)
result_df["PN AL78"] = result_df["PN AL78"].astype(str)

# Prepare lookup table (one price per PN)
dn_price_df = (
    ptc_normal_df
    .dropna(subset=["Unit Price"])
    .groupby("PN AL78", as_index=False)
    .agg({"Unit Price": "first"})
)

# Merge DN price
result_df = result_df.merge(
    dn_price_df,
    on="PN AL78",
    how="left"
)

# Rename column
result_df = result_df.rename(columns={"Unit Price": "DN Prc"})

# Move DN Prc after 'add'
dn_col = result_df.pop("DN Prc")
insert_pos = result_df.columns.get_loc("add") + 1
result_df.insert(insert_pos, "DN Prc", dn_col)


In [8]:
# =========================================================
# ADD EMPTY COLUMN 'OH AL78'
# =========================================================

insert_pos = result_df.columns.get_loc("DN Prc") + 1
result_df.insert(insert_pos, "OH AL78", "")


In [9]:
# =========================================================
# ADD COLUMN 'Cross to OH AL78'
# =========================================================

insert_pos = result_df.columns.get_loc("OH AL78") + 1

result_df.insert(
    insert_pos,
    "Cross to OH AL78",
    np.where(
        result_df["chk avail."] == "OK",
        "",
        np.where(
            (
                pd.to_numeric(result_df["Max.Req. CM"], errors="coerce").fillna(0)
              + pd.to_numeric(result_df["Max.Req. NM"], errors="coerce").fillna(0)
            )
            <= pd.to_numeric(result_df["OH AL78"], errors="coerce").fillna(0),
            "can take",
            "still need"
        )
    )
)

In [10]:
# =========================================================
# FINAL COLUMN ORDER (LOCKED – NOTHING DROPS)
# =========================================================
base_cols = [
    "PN AL78",
    "Long PN",
    "PN No suffix",
    "Description",
    "Avrg.OH CM",
    "Avrg. InRcv",
    "Sum Qty Resvrd",
    "Avrg. InTr",
    "Sum 7 Months Prior",
    "Sum OO CM",
    "Sum OO NM",
    "Sum OO N2M",
    "Max.Curr. Month",
    "Max.Req. CM",
    "Max.Req. NM",
]

final_cols = (
    base_cols
    + empty_cols
    + esd_periods
    + [
        "Missing",
        "Grand Total",
        "Balnc.",
        "chk Resvrd",
        "chk avail.",
        "add",
        "DN Prc",
        "OH AL78",
        "Cross to OH AL78",
    ]
)

# Keep only existing columns (extra safety)
final_cols = [c for c in final_cols if c in result_df.columns]

result_df = result_df.loc[:, final_cols]

In [11]:
# =========================================================
# critical
# =========================================================
result_df["critical"] = np.where(
    pd.to_numeric(result_df["Max.Curr. Month"], errors="coerce").fillna(0)
    >
    (
        pd.to_numeric(result_df["Avrg.OH CM"], errors="coerce").fillna(0)
      + pd.to_numeric(result_df["Avrg. InRcv"], errors="coerce").fillna(0)
      + pd.to_numeric(result_df["Avrg. InTr"], errors="coerce").fillna(0)
    ),
    1,
    2
)
# =========================================================
# MOVE 'critical' AFTER 'Cross to OH AL78'
# =========================================================
cols = list(result_df.columns)
idx = cols.index("Cross to OH AL78") + 1

cols.insert(idx, cols.pop(cols.index("critical")))
result_df = result_df[cols]


In [12]:
# =========================================================
# FIN BUSINESS CODE LOOKUP TABLE
# =========================================================
fin_code_df = (
    ptc_df[["PN AL78", "Fin Business Code"]]
    .dropna(subset=["PN AL78", "Fin Business Code"])
    .drop_duplicates(subset=["PN AL78"])
)
# =========================================================
# MERGE FIN BUSINESS CODE
# =========================================================
result_df = result_df.merge(
    fin_code_df,
    on="PN AL78",
    how="left"
)
# =========================================================
# INSERT BLANK COLUMN AFTER 'critical'
# =========================================================
result_df[" "] = ""   # blank column

cols = list(result_df.columns)

# move blank column after 'critical'
crit_idx = cols.index("critical") + 1
cols.insert(crit_idx, cols.pop(cols.index(" ")))

# move 'Fin Business Code' after the blank column
blank_idx = cols.index(" ") + 1
cols.insert(blank_idx, cols.pop(cols.index("Fin Business Code")))

result_df = result_df[cols]


In [13]:
# engine2_df = pd.read_excel("engine model PTC.xlsx",sheet_name="Sheet2")
# engine_df = pd.read_excel("engine model draft.xlsx")
engine_new_df = pd.read_excel("Engine Classification 2 Maret 26.xlsx")

In [14]:
print(result_df)

          PN AL78      Long PN  PN No suffix             Description  \
0          101843    010184300        101843                    SHIM   
1          103023    010302300        103023  SCREW,HEXAGON HEAD CAP   
2          107460    010746000        107460                    CLIP   
3          107695    010769500        107695                PIN,ROLL   
4          108722    010872200        108722                    CLIP   
..            ...          ...           ...                     ...   
676       S   306    S00030600       S   306      KEY,PLAIN WOODRUFF   
677       S   962    S00096200       S   962               PLUG,PIPE   
678  S   962    E  S00096200 E  S   962    E               DRAINCOCK   
679  S  1040    A  S00104000 A  S  1040    A      ELBOW,MALE ADAPTER   
680       S  2268    S00226800       S  2268          FITTING,GREASE   

     Avrg.OH CM  Avrg. InRcv  Sum Qty Resvrd  Avrg. InTr  Sum 7 Months Prior  \
0           0.0          0.0               0         2.

In [15]:
# Clean PN columns
engine_new_df["PN No Suffix"] = engine_new_df["PN No Suffix"].astype(str).str.strip()
result_df["PN No suffix"] = result_df["PN No suffix"].astype(str).str.strip()

# Create lookup table (merge multiple engine rows per PN if needed)
engine_lookup = (
    engine_new_df.groupby("PN No Suffix")["Engine Classification"]
    .apply(lambda x: ", ".join(dict.fromkeys(
        e.strip() for val in x.dropna() for e in val.split(",")
    )))
    .reset_index()
    .rename(columns={"Engine Classification": "Engine Model"})
)

# Merge into result_df
result_df = result_df.merge(
    engine_lookup,
    left_on="PN No suffix",      # from result_df
    right_on="PN No Suffix",     # from engine_lookup
    how="left"
)


In [16]:
print(result_df)

          PN AL78      Long PN  PN No suffix             Description  \
0          101843    010184300        101843                    SHIM   
1          103023    010302300        103023  SCREW,HEXAGON HEAD CAP   
2          107460    010746000        107460                    CLIP   
3          107695    010769500        107695                PIN,ROLL   
4          108722    010872200        108722                    CLIP   
..            ...          ...           ...                     ...   
676       S   306    S00030600       S   306      KEY,PLAIN WOODRUFF   
677       S   962    S00096200       S   962               PLUG,PIPE   
678  S   962    E  S00096200 E  S   962    E               DRAINCOCK   
679  S  1040    A  S00104000 A  S  1040    A      ELBOW,MALE ADAPTER   
680       S  2268    S00226800       S  2268          FITTING,GREASE   

     Avrg.OH CM  Avrg. InRcv  Sum Qty Resvrd  Avrg. InTr  Sum 7 Months Prior  \
0           0.0          0.0               0         2.

In [17]:
final_columns = [
    "PN AL78",
    "Long PN",
    "Description",
    "Avrg.OH CM",
    "Avrg. InRcv",
    "Sum Qty Resvrd",
    "Avrg. InTr",
    "Sum 7 Months Prior",
    "Sum OO CM",
    "Sum OO NM",
    "Sum OO N2M",
    "Max.Curr. Month",
    "Max.Req. CM",
    "Max.Req. NM",

    # ===== 6 BLANK COLUMNS =====
    "Blank_1",
    "Blank_2",
    "Blank_3",
    "Blank_4",
    "Blank_5",
    "Blank_6",

    "PN No suffix",
]

# 🔹 INSERT ESD MONTHS HERE (DYNAMIC)
final_columns.extend(esd_periods)

final_columns.extend([
    "Missing",
    "Grand Total",

    # ===== 2 BLANK COLUMNS =====
    "Blank_7",
    "Blank_8",

    "chk avail.",
    "Balnc.",
    "chk Resvrd",
    "add",
    "DN Prc",
    "OH AL78",
    "Cross to OH AL78",
    "critical",

    # ===== 1 BLANK COLUMN =====
    "Blank_9",

    "Fin Business Code",
    "Engine Model",
])

for col in final_columns:
    if col not in result_df.columns:
        result_df[col] = ""

result_df["Missing"] = pd.to_numeric(result_df["Missing"], errors="coerce").fillna(0)
result_df = result_df[final_columns]


In [18]:
print(result_df)

          PN AL78      Long PN             Description  Avrg.OH CM  \
0          101843    010184300                    SHIM         0.0   
1          103023    010302300  SCREW,HEXAGON HEAD CAP         5.0   
2          107460    010746000                    CLIP        56.0   
3          107695    010769500                PIN,ROLL         7.0   
4          108722    010872200                    CLIP       118.0   
..            ...          ...                     ...         ...   
676       S   306    S00030600      KEY,PLAIN WOODRUFF         2.0   
677       S   962    S00096200               PLUG,PIPE         9.0   
678  S   962    E  S00096200 E               DRAINCOCK        61.0   
679  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER         0.0   
680       S  2268    S00226800          FITTING,GREASE        12.0   

     Avrg. InRcv  Sum Qty Resvrd  Avrg. InTr  Sum 7 Months Prior  Sum OO CM  \
0            0.0               0         2.0                   8          0   
1

In [19]:
# =========================================================
# EXPORT
# =========================================================
result_df.to_excel("Pvt Final draft.xlsx", index=False)